# EDA: PaySim Synthetic Financial Transactions

Exploratory Data Analysis of the PaySim dataset for fraud detection patterns.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum, avg as _avg, desc, when, round

spark = SparkSession.builder.getOrCreate()
df = spark.read.csv("data/sample_paysim.csv", header=True, inferSchema=True)
print(f"Record count: {df.count()}")
df.printSchema()

In [ ]:
print("Transaction Type Distribution:")
df.groupBy("type").count().orderBy(desc("count")).show(truncate=False)

print("Fraud vs Legitimate:")
df.groupBy("isFraud").count().show()

In [ ]:
print("Amount Statistics by Transaction Type:")
df.groupBy("type").agg(
    _sum("amount").alias("total_amount"),
    _avg("amount").alias("avg_amount"),
    count("*").alias("count")
).orderBy(desc("total_amount")).show(truncate=False)

In [ ]:
print("Fraud Rate by Transaction Type:")
df.groupBy("type").agg(
    count("*").alias("total"),
    _sum(when(col("isFraud") == 1, 1).otherwise(0)).alias("fraud_count"),
    round(_sum(when(col("isFraud") == 1, 1).otherwise(0)) / count("*"), 4).alias("fraud_rate")
).orderBy(desc("fraud_rate")).show(truncate=False)

In [ ]:
print("High-Value Transactions (amount >= 10k):")
high_value = df.filter(col("amount") >= 10000)
high_value.groupBy("isFraud").count().show()

print("Flagged vs Actual Fraud:")
df.groupBy("isFlaggedFraud", "isFraud").count().orderBy("isFlaggedFraud", "isFraud").show()